# 🚇 서울 지하철 혼잡도 & 사고 분석 (1~9호선) — 정책입안자 대상
**SCQA · 민토 피라미드 · 기초통계 · 시각화 → 제안 + Action Plan**

| SCQA | 내용 |
|---|---|
| **S 상황** | 코로나 후 수요 회복(2024년 91%)으로 출퇴근 혼잡 재심화 |
| **C 문제** | 이미 혼잡한 노선·역·시간대에 **사고·급정차가 겹치면 혼잡 폭증(안전 위험)**. 5년 사고 2,837건 분석 → 08시 최다·출퇴근 34.4% 집중 |
| **Q 질문** | 어느 노선·역·시간대가 **"사고 시 가장 위험한 취약 지점"**인가? |
| **A 답** | 2호선·도심 업무지구·출퇴근에 혼잡·사고 동시 집중 → **취약 지점 우선 투자 + 사고·수요 대응** |

> 범위: 서울 1~9호선 / "혼잡"=역 승하차 인원 / 단면=최근월·추이=전체

## 0. 환경 설정 & 데이터 로드

In [ ]:
# (Colab 최초 1회) 한글 폰트
!sudo apt-get install -y -qq fonts-nanum >/dev/null 2>&1
!fc-cache -fv >/dev/null 2>&1
!rm -rf ~/.cache/matplotlib

In [ ]:
import pandas as pd, numpy as np, re
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumBarunGothic'   # 로컬(mac)이면 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 파일 자동 탐색 (로컬/Colab 어디에 올리든 OK)
import os
def D(fn):
    for q in [fn, 'data/'+fn, '/content/'+fn, '/content/data/'+fn]:
        if os.path.exists(q): return q
    return fn

In [ ]:
# 승하차 데이터 (★ cp949)
df = pd.read_csv(D('서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv'), encoding='cp949')
print('원본:', df.shape)

## 1. 데이터 정제 (총승하차 + 중복제거 + 1~9호선)

In [ ]:
승차컬=[c for c in df.columns if c.endswith('승차인원')]
하차컬=[c for c in df.columns if c.endswith('하차인원')]
df['총승차']=df[승차컬].sum(axis=1); df['총하차']=df[하차컬].sum(axis=1)
df['총승하차']=df['총승차']+df['총하차']
df=df.drop_duplicates(subset=['사용월','호선명','지하철역'])
df['연도']=df['사용월']//100
df=df[df['호선명'].str.match(r'^[1-9]호선')].copy()
df['호선']=df['호선명'].str.extract(r'^([1-9]호선)')
LATEST=df['사용월'].max(); cur=df[df['사용월']==LATEST].copy()
print('정제 후:', df.shape, '| 기준월:', LATEST, f"({cur['지하철역'].nunique()}역)")

## 2. 근거 1 — 연도별 추이 (코로나 충격/회복)

In [ ]:
yr=df.groupby('연도')['총승하차'].sum()/1e8
print(yr.round(1)); print(f"2020 {(yr[2020]/yr[2019]-1)*100:+.1f}% | 2024 {(yr[2024]/yr[2019]-1)*100:+.1f}%")
fig,ax=plt.subplots(figsize=(9,4)); yr.plot(marker='o',ax=ax,color='#E63946',lw=2.5)
ax.axvspan(2019.7,2020.5,alpha=.12,color='gray'); ax.set_title('연도별 총 이용객'); ax.set_ylabel('억 명'); ax.grid(alpha=.3); plt.show()

## 3. 근거 2 — 호선별 차이 (ANOVA)

In [ ]:
line=cur.groupby('호선')['총승하차'].agg(총합='sum',역당평균='mean',역수='count').sort_values('총합',ascending=False)
print(line.round(0))
g=[x['총승하차'].values for _,x in cur.groupby('호선') if len(x)>=5]
F,p=stats.f_oneway(*g); print(f"ANOVA F={F:.1f}, p={p:.2e}")
fig,ax=plt.subplots(figsize=(9,4)); (line['총합']/1e6)[::-1].plot.barh(ax=ax,color=['#999']*8+['#E63946'])
ax.set_title('호선별 총 이용객 (2호선 압도)'); ax.set_xlabel('백만'); plt.show()

## 4. 근거 3 — 시간대 출퇴근 피크 & 유입/유출형

In [ ]:
승t=cur[승차컬].sum(); 하t=cur[하차컬].sum()
print('승차피크',승t.idxmax(),'| 하차피크',하t.idxmax())
am승=['07시-08시 승차인원','08시-09시 승차인원']; am하=['07시-08시 하차인원','08시-09시 하차인원']
cur['유입비']=cur[am하].sum(axis=1)/(cur[am승].sum(axis=1)+1)
big=cur[cur[am승].sum(axis=1)>5000]
print('유입형(업무):',list(big.nlargest(5,'유입비')['지하철역']))
print('유출형(주거):',list(big.nsmallest(5,'유입비')['지하철역']))
fig,ax=plt.subplots(figsize=(11,4)); x=range(len(승차컬))
ax.plot(x,승t.values/1e6,'o-',label='승차',color='#457B9D'); ax.plot(x,하t.values/1e6,'s-',label='하차',color='#E63946')
ax.set_xticks(x); ax.set_xticklabels([c.split('시-')[0] for c in 승차컬],rotation=90,fontsize=7)
ax.set_title('시간대별 승하차'); ax.legend(); ax.grid(alpha=.3); plt.show()

## 5. 근거 4 — 역별 혼잡 Top & 집중도 (+ 지도)

In [ ]:
top=cur.nlargest(10,'총승하차')[['지하철역','호선','총승하차']]; print(top.to_string(index=False))
s=cur['총승하차'].sort_values(ascending=False).values; n=max(1,len(s)//10)
print(f"상위10%({n}역)={s[:n].sum()/s.sum()*100:.1f}% 집중")
# 지도는 좌표 파일(data/도시철도역사정보_좌표.xlsx) + folium 사용 → charts/지하철_혼잡_지도.html 참고

## 6. 근거 5 — 사고도 출퇴근·혼잡역에 몰린다 ⭐
서울교통공사 5년 사고 2,837건(발생시간 포함) 직접 분석 → 뉴스가 아닌 데이터로 입증.

In [ ]:
acc=pd.read_csv(D('서울교통공사_사고현황_5년.csv'), encoding='cp949')  # Colab: 업로드
acc['시']=acc['발생시간'].astype(str).str.extract(r'^(\d{1,2}):').astype(float)
acc=acc.dropna(subset=['시']); acc['시']=acc['시'].astype(int); N=len(acc)
hc=acc['시'].value_counts().sort_index()
peak=acc['시'].isin([7,8,18,19]).sum()
print(f"총 {N}건 | 최다 {hc.idxmax()}시 {hc.max()}건 | 출퇴근(7·8·18·19시) {peak}건 = {peak/N*100:.1f}% (균등이면 ~21%)")
print('사고 다발역:', list(acc['발생역'].value_counts().head(5).index))
print('연도별:', acc.assign(y=pd.to_datetime(acc['발생일'],errors='coerce').dt.year)['y'].value_counts().sort_index().to_dict())
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(hc.index,hc.values,color=['#E63946' if h in[7,8,18,19] else '#90A4AE' for h in hc.index])
ax.set_title(f'시간대별 지하철 사고 ({N}건) — 출퇴근(빨강) 집중'); ax.set_xlabel('시'); ax.set_ylabel('건수'); ax.grid(alpha=.2,axis='y'); plt.show()

## 7. 🏛️ 핵심 결론 → 제안 → Action Plan (정책입안자 / 투트랙)

> **핵심 결론**: 서울 지하철은 **2호선·도심 업무지구·출퇴근(08·18시)**에 혼잡이 구조적으로 집중되고, **사고도 같은 시간대·혼잡 환승역에 몰린다**(2,837건 중 출퇴근 34.4%, 08시 최다). 코로나 후 재혼잡 + 사고 증가(+81%)로 위험이 커지는 중.

**근거**: ①2020 –27%→재혼잡 ②2호선 압도(ANOVA p<0.001) ③출근 하차 08시·퇴근 승차 18시 ④상위10% 역=28.5% ⑤**사고 08시 최다·출퇴근 34.4%·다발역=혼잡역**

| 트랙 | 제안 | Action Plan |
|---|---|---|
| 🚇 서울교통공사 | 집중역 우선 관리 + **사고 비상 혼잡 대응** | 단기 혼잡·사고 실시간 안내·인력 / 상시 신호·전기 설비 예방정비 / 중기 2호선·집중역 동선 개선 |
| 🏛️ 국회 국토위 | 수요 분산 + **노후 설비 예산** | 시차출퇴근 인센티브 / 노후 신호·전동차 교체 예산 / 안전기준 법제화 |

*범위: 서울 1~9호선 / 혼잡=승하차 / 사고=5년 2,837건*
